Set runtime to T4 GPU, then Runtime → Run all.

In [ ]:
!pip install -q mpich

## 1. Array addition (CUDA)

In [ ]:
%%bash
cat > /tmp/p1_array_add.cu << 'EOF'
// Program 1: Addition of two arrays using CUDA
#include <stdio.h>

#define N 10

__global__ void add(int *a, int *b, int *c)
{
    int i = threadIdx.x;              // each thread adds one element
    
    c[i] = a[i] + b[i];
}

int main()
{
    int a[N], b[N], c[N];
    int *d_a, *d_b, *d_c;             // d_ = device (GPU) memory
    int size = N * sizeof(int);

    for (int i = 0; i < N; i++) {
        a[i] = i;
        b[i] = i * 2;
    }

    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    add<<<1, N>>>(d_a, d_b, d_c);     // 1 block, N threads

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++)
        printf("%d + %d = %d\n", a[i], b[i], c[i]);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}
EOF
nvcc -arch=sm_75 /tmp/p1_array_add.cu -o /tmp/p1 && /tmp/p1

## 2. Matrix multiplication (OpenMP)

In [ ]:
%%bash
gcc -fopenmp -x c - -o /tmp/p2 << 'EOF'
// Program 2: Parallel matrix multiplication using OpenMP
#include <stdio.h>
#include <omp.h>

#define N 500

int A[N][N], B[N][N], C[N][N];

void multiply()
{
    #pragma omp parallel for          // rows of C are split among threads
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int sum = 0;

            for (int k = 0; k < N; k++)
                sum += A[i][k] * B[k][j];

            C[i][j] = sum;
        }
    }
}

int main()
{
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++) 
        {
            A[i][j] = 1;
            B[i][j] = 2;
        }
    }

    for (int t = 1; t <= 8; t *= 2) 
    {
        omp_set_num_threads(t);

        double start = omp_get_wtime();
        multiply();
        double time = omp_get_wtime() - start;

        printf("Threads = %d   Time = %f s\n", t, time);
    }

    printf("C[0][0] = %d (expected %d)\n", C[0][0], 2 * N);

    return 0;
}
EOF
/tmp/p2

## 3. Merge sort (OpenMP)

In [ ]:
%%bash
gcc -fopenmp -x c - -o /tmp/p3 << 'EOF'
// Program 3: Parallel merge sort using OpenMP tasks
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

#define N 2000000

int orig[N], a[N], tmp[N];

void merge(int left, int mid, int right)
{
    int i = left, j = mid + 1, k = left;

    while (i <= mid && j <= right)
    {
        if (a[i] <= a[j])
        {
            tmp[k] = a[i];
            i++;
        }
        else
        {
            tmp[k] = a[j];
            j++;
        }

        k++;
    }
    
    while (i <= mid)
    {
        tmp[k] = a[i];
        i++; k++;
    }
    
    while (j <= right)
    {
        tmp[k] = a[j];
        j++; k++;
    }
    
    for (i = left; i <= right; i++) a[i] = tmp[i];
}

void mergeSort(int left, int right)
{
    if (left >= right) return;

    int mid = (left + right) / 2;

    #pragma omp task      // each half becomes a parallel task
    mergeSort(left, mid);

    #pragma omp task
    mergeSort(mid + 1, right);

    #pragma omp taskwait  // wait for both halves
    merge(left, mid, right);
}

void reset()                             // give every run the same unsorted data
{
    for (int i = 0; i < N; i++) a[i] = orig[i];
}

int main()
{
    for (int i = 0; i < N; i++) orig[i] = rand();

    // Sequential: outside a parallel region there is only 1 thread
    reset();
    double start = omp_get_wtime();
    mergeSort(0, N - 1);
    double seq = omp_get_wtime() - start;
    printf("Sequential time = %f s\n", seq);

    // Parallel
    reset();
    start = omp_get_wtime();
    #pragma omp parallel
    {
        #pragma omp single               // one thread starts the recursion
        mergeSort(0, N - 1);
    }
    double par = omp_get_wtime() - start;
    
    printf("Parallel time   = %f s\n", par);
    printf("Speedup         = %.2f\n", seq / par);

    return 0;
}
EOF
/tmp/p3

## 4. threadprivate and copyin (OpenMP)

In [ ]:
%%bash
gcc -fopenmp -x c - -o /tmp/p4 << 'EOF'
// Program 4: threadprivate and copyin in OpenMP
#include <stdio.h>
#include <omp.h>

int x;                          // global variable
#pragma omp threadprivate(x)    // every thread gets its own private copy of x

int main()
{
    omp_set_num_threads(4);
    x = 10;                     // only the master thread's copy becomes 10

    printf("Region 1 (without copyin): other threads still have x = 0\n");
    #pragma omp parallel
    {
        printf("  Thread %d: x = %d\n", omp_get_thread_num(), x);
    }

    printf("\nRegion 2 (with copyin): master's x is copied to all threads\n");
    #pragma omp parallel copyin(x)
    {
        int id = omp_get_thread_num();
        printf("  Thread %d: x = %d\n", id, x);
        x = x + id;             // each thread changes only its own copy
    }

    printf("\nRegion 3: each thread's copy keeps its value from region 2\n");
    #pragma omp parallel
    {
        printf("  Thread %d: x = %d\n", omp_get_thread_num(), x);
    }

    printf("\nMaster thread after all regions: x = %d\n", x);
    return 0;
}
EOF
/tmp/p4

## 5. Loop scheduling (OpenMP)

In [ ]:
%%bash
gcc -fopenmp -x c - -o /tmp/p5 << 'EOF'
// Program 5: OpenMP loop scheduling - static, dynamic, guided
#include <stdio.h>
#include <omp.h>

#define N 16

void work(int i)
{
    double s = 0;

    int n = (i + 1) * 5000000;          // later iterations do MORE work

    for (int j = 0; j < n; j++)
        s += 1;
    
    printf("Iteration %2d -> Thread %d\n", i, omp_get_thread_num());
}

int main()
{
    double start;
    omp_set_num_threads(4);

    printf("STATIC (equal blocks of iterations, fixed in advance):\n");
    start = omp_get_wtime();
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < N; i++) work(i);
    printf("Time = %f s\n\n", omp_get_wtime() - start);

    printf("DYNAMIC (free thread takes the next iteration):\n");
    start = omp_get_wtime();
    #pragma omp parallel for schedule(dynamic)
    for (int i = 0; i < N; i++) work(i);
    printf("Time = %f s\n\n", omp_get_wtime() - start);

    printf("GUIDED (big chunks first, then smaller chunks):\n");
    start = omp_get_wtime();
    #pragma omp parallel for schedule(guided)
    for (int i = 0; i < N; i++) work(i);
    printf("Time = %f s\n", omp_get_wtime() - start);

    return 0;
}
EOF
/tmp/p5

## 6. Prime numbers (OpenMP)

In [ ]:
%%bash
gcc -fopenmp -x c - -o /tmp/p6 << 'EOF'
// Program 6: Parallel prime number generation using OpenMP
#include <stdio.h>
#include <omp.h>

#define N 10000

int isPrime(int n)
{
    if (n <= 1) return 0;

    int count = 0;

    for (int i = 2; i <= n; i++)
    {
        if (n % i == 0) count++;
    }
    
    return (count == 1);
}

int main()
{
    // Sequential
    int count = 0;

    double start = omp_get_wtime();
    for (int i = 2; i <= N; i++)
        count += isPrime(i);
    double seq = omp_get_wtime() - start;
    
    printf("Sequential: primes up to %d = %d, time = %f s\n\n", N, count, seq);

    // Parallel
    for (int t = 1; t <= 8; t *= 2)
    {
        count = 0;

        omp_set_num_threads(t);

        start = omp_get_wtime();
        #pragma omp parallel for reduction(+:count)
        for (int i = 2; i <= N; i++)
            count += isPrime(i);
        double time = omp_get_wtime() - start;

        double speedup = seq / time;

        printf("Threads: %d\tTime: %f\tSpeedup: %.2f\tEfficiency: %.2f\n", t, time, speedup, speedup / t);
    }

    printf("\nPrimes found (parallel) = %d\n", count);
    
    return 0;
}
EOF
/tmp/p6

## 7. Array sum with Scatter and Reduce (MPI)

In [ ]:
%%bash
mpicc -x c - -o /tmp/p7 << 'EOF'
// Program 7: Parallel array summation using MPI_Scatter and MPI_Reduce
#include <stdio.h>
#include <mpi.h>

#define N 1000000          // must be divisible by the number of processes

int a[N], part[N];

int main(int argc, char *argv[])
{
    int rank, size;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int chunk = N / size;

    if (rank == 0)                       // root fills the array
    {
        for (int i = 0; i < N; i++)
        {
            a[i] = 1;
        }
    }

    // Root sends one chunk to every process (including itself)
    MPI_Scatter(a, chunk, MPI_INT, part, chunk, MPI_INT, 0, MPI_COMM_WORLD);

    int local = 0;
    for (int i = 0; i < chunk; i++) 
    {
        local += part[i];
    }
    printf("Process %d: partial sum = %d\n", rank, local);

    // Add all partial sums into total on root
    int total;
    MPI_Reduce(&local, &total, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0)
        printf("Total sum = %d (expected %d)\n", total, N);

    MPI_Finalize();
    return 0;
}
EOF
mpirun -np 4 /tmp/p7

## 8. Send/Recv and Bcast (MPI)

In [ ]:
%%bash
mpicc -x c - -o /tmp/p8 << 'EOF'
// Program 8: Point-to-point (MPI_Send/MPI_Recv) and broadcast (MPI_Bcast)
#include <stdio.h>
#include <string.h>
#include <mpi.h>

int main(int argc, char *argv[])
{
    int rank, size;
    char msg[20];
    int arr[5] = {0, 0, 0, 0, 0};

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    // 1. Point-to-point: process 0 sends a string to process 1
    if (rank == 0) {
        strcpy(msg, "Hello from P0");
        MPI_Send(msg, 20, MPI_CHAR, 1, 0, MPI_COMM_WORLD);
        printf("Process 0 sent: %s\n", msg);
    } else if (rank == 1) {
        MPI_Recv(msg, 20, MPI_CHAR, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("Process 1 received: %s\n", msg);
    }

    // 2. Broadcast: process 0 sends an array to ALL processes
    if (rank == 0)
        for (int i = 0; i < 5; i++) arr[i] = (i + 1) * 10;

    MPI_Bcast(arr, 5, MPI_INT, 0, MPI_COMM_WORLD);   // called by every process

    printf("Process %d after Bcast: %d %d %d %d %d\n",
           rank, arr[0], arr[1], arr[2], arr[3], arr[4]);

    MPI_Finalize();
    return 0;
}
EOF
mpirun -np 4 /tmp/p8

## 9. GPU properties and kernel launch (CUDA)

In [ ]:
%%bash
cat > /tmp/p9_cuda_props.cu << 'EOF'
// Program 9: GPU properties + kernel launch with different blocks/threads
#include <stdio.h>

__global__ void hello()
{
    int id = blockIdx.x * blockDim.x + threadIdx.x;   // global thread id
    printf("Block %d, Thread %d, Global ID %d\n", blockIdx.x, threadIdx.x, id);
}

int main()
{
    cudaDeviceProp p;
    cudaGetDeviceProperties(&p, 0);

    printf("GPU Name             : %s\n", p.name);
    printf("Compute Capability   : %d.%d\n", p.major, p.minor);
    printf("Multiprocessors (SM) : %d\n", p.multiProcessorCount);
    printf("Warp Size            : %d\n", p.warpSize);
    printf("Max Threads/Block    : %d\n", p.maxThreadsPerBlock);

    printf("\n<<<1 block, 4 threads>>>\n");
    hello<<<1, 4>>>();
    cudaDeviceSynchronize();          // wait for GPU so its printf output appears

    printf("\n<<<2 blocks, 4 threads>>>\n");
    hello<<<2, 4>>>();
    cudaDeviceSynchronize();

    printf("\n<<<4 blocks, 2 threads>>>\n");
    hello<<<4, 2>>>();
    cudaDeviceSynchronize();

    return 0;
}
EOF
nvcc -arch=sm_75 /tmp/p9_cuda_props.cu -o /tmp/p9 && /tmp/p9

## 10. Vector addition, CPU vs GPU (CUDA)

In [ ]:
%%bash
cat > /tmp/p10_vec_add_time.cu << 'EOF'
// Program 10: Vector addition - CPU vs GPU execution time
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 10000000

__global__ void add(float *a, float *b, float *c)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // global thread id
    if (i < N)
        c[i] = a[i] + b[i];
}

int main()
{
    int size = N * sizeof(float);

    float *a = (float*)malloc(size);
    float *b = (float*)malloc(size);
    float *c = (float*)malloc(size);      // CPU result
    float *g = (float*)malloc(size);      // GPU result
    
    float *d_a, *d_b, *d_c;

    for (int i = 0; i < N; i++) {
        a[i] = i;
        b[i] = i * 2;
    }

    // CPU
    clock_t start = clock();
    for (int i = 0; i < N; i++)
        c[i] = a[i] + b[i];
    double cpu = (double)(clock() - start) / CLOCKS_PER_SEC;

    // GPU
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = N / threads + 1;  // enough blocks to cover N

    start = clock();
    add<<<blocks, threads>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();                   // wait for GPU to finish
    double gpu = (double)(clock() - start) / CLOCKS_PER_SEC;

    cudaMemcpy(g, d_c, size, cudaMemcpyDeviceToHost);

    printf("Results:\n");
    printf("CPU time = %f s\n", cpu);
    printf("GPU time = %f s\n", gpu);
    printf("Speedup  = %.2f\n", cpu / gpu);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(a); free(b); free(c); free(g);
    return 0;
}
EOF
nvcc -arch=sm_75 /tmp/p10_vec_add_time.cu -o /tmp/p10 && /tmp/p10

## 11. Linear regression (PyTorch)

In [ ]:
# Program 11: Linear Regression using PyTorch (gradients + optimization)
import torch
import torch.nn as nn

torch.manual_seed(0)

# Data following y = 2x + 1
x = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])
y = 2 * x + 1

model = nn.Linear(1, 1)                       # y = w*x + b
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(1000):
    pred = model(x)
    loss = loss_fn(pred, y)

    optimizer.zero_grad()                     # clear old gradients
    loss.backward()                           # compute dLoss/dw and dLoss/db
    optimizer.step()                          # w = w - lr * grad

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}  Loss = {loss.item():.4f}  "
              f"dL/dw = {model.weight.grad.item():.4f}  dL/db = {model.bias.grad.item():.4f}")

print(f"\nLearned: w = {model.weight.item():.3f}, b = {model.bias.item():.3f}  (true: w = 2, b = 1)")

## 12. Feedforward neural network (PyTorch)

In [ ]:
# Program 12: Basic feedforward neural network on random data using PyTorch
import torch
import torch.nn as nn

torch.manual_seed(0)

# Random data: 100 samples, 10 input features, 1 output
X = torch.randn(100, 10)
y = torch.randn(100, 1)

model = nn.Sequential(
    nn.Linear(10, 16),     # input layer -> hidden layer
    nn.ReLU(),             # activation
    nn.Linear(16, 1)       # hidden layer -> output
)
print(model)

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    pred = model(X)                 # forward pass
    loss = loss_fn(pred, y)

    optimizer.zero_grad()
    loss.backward()                 # backward pass
    optimizer.step()                # update weights

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}  Loss = {loss.item():.4f}")

print(f"Final Loss = {loss.item():.4f}")